# LinkedIn CS Job Market Analysis
**Dataset:** [arshkon/linkedin-job-postings](https://www.kaggle.com/datasets/arshkon/linkedin-job-postings) (2023–2024, 124k+ postings)

**Sections:**
0. Data Loading & Cleaning
1. CS Job Filtering & Breakdown
2. Job Listing Trends Over Time
3. Skills Analysis
4. Salary & Compensation
5. Work Type & Remote Trends
6. Experience Level & Geography
7. Company & Demand Stats

## 0. Data Loading & Cleaning

In [ ]:
import re
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Ensures charts render inline in VS Code, JupyterLab, and classic Notebook
pio.renderers.default = "notebook"

DATA_DIR = "../data"

# ── Load files ────────────────────────────────────────────────────────────────
jobs      = pd.read_csv(f"{DATA_DIR}/postings.csv", low_memory=False)
companies = pd.read_csv(f"{DATA_DIR}/companies/companies.csv", low_memory=False)
job_skills_raw = pd.read_csv(f"{DATA_DIR}/jobs/job_skills.csv")
skills_map     = pd.read_csv(f"{DATA_DIR}/mappings/skills.csv")

print(f"Postings : {jobs.shape}")
print(f"Companies: {companies.shape}")
print(f"Job-skill pairs: {job_skills_raw.shape}")
jobs.head(3)

In [ ]:
# ── Parse timestamps ─────────────────────────────────────────────────────────
jobs["listed_dt"] = pd.to_datetime(jobs["listed_time"], unit="ms", utc=True).dt.tz_localize(None)
jobs["expiry_dt"] = pd.to_datetime(jobs["expiry"],      unit="ms", utc=True, errors="coerce").dt.tz_localize(None)
jobs["month"] = jobs["listed_dt"].dt.to_period("M").astype(str)
jobs["week"]  = jobs["listed_dt"].dt.to_period("W").astype(str)

# ── Drop rows missing critical fields ────────────────────────────────────────
jobs = jobs.dropna(subset=["title", "listed_time"])
jobs["title_lower"]  = jobs["title"].str.lower()
jobs["skills_desc"]  = jobs["skills_desc"].fillna("").str.lower()

# ── Merge company details ─────────────────────────────────────────────────────
jobs = jobs.merge(
    companies[["company_id", "company_size", "state", "city"]],
    on="company_id", how="left"
)

print(f"Date range : {jobs['listed_dt'].min().date()} → {jobs['listed_dt'].max().date()}")
print(f"Clean count: {len(jobs):,}")

## 1. CS Job Filtering & Subcategory Breakdown

In [ ]:
CS_CATEGORIES = {
    "SWE": [
        "software engineer", "software developer", "backend", "back-end",
        "frontend", "front-end", "full stack", "fullstack", "full-stack",
        "web developer", "mobile developer", "ios developer", "android developer",
        "application developer", "programmer"
    ],
    "Data": [
        "data engineer", "data analyst", "data scientist", "analytics engineer",
        "business intelligence", "etl developer", "data warehouse", "data architect"
    ],
    "ML / AI": [
        "machine learning", "deep learning", "nlp engineer", "computer vision",
        "artificial intelligence", "ai engineer", "ai/ml", "ml engineer",
        "mlops", "llm", "generative ai", "research scientist"
    ],
    "Cloud / Infra": [
        "devops", "site reliability", "sre", "cloud engineer", "platform engineer",
        "infrastructure engineer", "systems engineer", "network engineer",
        "database administrator", "systems administrator"
    ],
    "Security": [
        "security engineer", "cybersecurity", "information security",
        "application security", "appsec", "penetration tester",
        "security analyst", "soc analyst"
    ],
    "Other CS": [
        "technical program manager", "technical product manager",
        "solutions architect", "solutions engineer", "qa engineer",
        "quality assurance engineer", "embedded engineer", "firmware engineer",
        "it manager", "computer scientist"
    ],
}

def classify_job(title: str) -> str:
    for category, keywords in CS_CATEGORIES.items():
        for kw in keywords:
            if kw in title:
                return category
    return "Non-CS"

jobs["category"] = jobs["title_lower"].apply(classify_job)
cs = jobs[jobs["category"] != "Non-CS"].copy()

print(f"CS jobs: {len(cs):,} / {len(jobs):,} ({len(cs)/len(jobs):.1%})")
jobs["category"].value_counts()

In [ ]:
cat_counts = jobs["category"].value_counts().reset_index()

fig = px.bar(
    cat_counts,
    x="category", y="count",
    color="category",
    title="Job Postings by CS Subcategory (all postings)",
    labels={"category": "Category", "count": "Postings"},
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
cs_counts = cs["category"].value_counts().reset_index()

fig = px.pie(
    cs_counts,
    names="category", values="count",
    title="CS Job Postings — Subcategory Share",
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig.show()

## 2. Job Listing Trends Over Time

In [ ]:
monthly = (
    cs.groupby(["month", "category"])
      .size()
      .reset_index(name="count")
      .sort_values("month")
)

fig = px.line(
    monthly,
    x="month", y="count",
    color="category",
    markers=True,
    title="Monthly CS Job Listings by Subcategory",
    labels={"month": "Month", "count": "Job Postings", "category": "Category"},
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
# 4-week rolling average for total CS listings
weekly_total = (
    cs.groupby("week")
      .size()
      .reset_index(name="count")
      .sort_values("week")
)
weekly_total["rolling_avg"] = weekly_total["count"].rolling(4, min_periods=1).mean()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=weekly_total["week"], y=weekly_total["count"],
    name="Weekly Count", marker_color="lightsteelblue", opacity=0.6
))
fig.add_trace(go.Scatter(
    x=weekly_total["week"], y=weekly_total["rolling_avg"],
    name="4-Week Rolling Avg", line=dict(color="royalblue", width=2.5)
))
fig.update_layout(
    title="Weekly CS Job Listings with 4-Week Rolling Average",
    xaxis_title="Week", yaxis_title="Job Postings",
    xaxis_tickangle=45
)
fig.show()

In [ ]:
# Stacked area: subcategory share per month
monthly_pct = monthly.copy()
total_per_month = monthly_pct.groupby("month")["count"].transform("sum")
monthly_pct["pct"] = monthly_pct["count"] / total_per_month * 100

fig = px.area(
    monthly_pct,
    x="month", y="pct",
    color="category",
    title="Monthly Share of CS Subcategories (%)",
    labels={"month": "Month", "pct": "Share (%)", "category": "Category"},
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig.update_xaxes(tickangle=45)
fig.show()

## 3. Skills Analysis
### 3a. LinkedIn Skill Categories (from job_skills.csv)

In [ ]:
# Join job_skills with skill names, then with CS postings
job_skills = (
    job_skills_raw
      .merge(skills_map, on="skill_abr", how="left")
      .merge(cs[["job_id", "category", "month"]], on="job_id", how="inner")
)

print(f"CS job-skill pairs: {len(job_skills):,}")
job_skills.head()

In [ ]:
# Top LinkedIn skill categories across all CS jobs
top_li_skills = (
    job_skills["skill_name"]
      .value_counts()
      .reset_index()
)

fig = px.bar(
    top_li_skills[::-1],
    x="count", y="skill_name",
    orientation="h",
    title="LinkedIn Skill Categories Across CS Job Postings",
    labels={"skill_name": "Skill Category", "count": "Postings"},
    color="count",
    color_continuous_scale="Blues"
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [ ]:
# Skill category heatmap: rate per CS subcategory
cat_totals = cs["category"].value_counts().to_dict()

heat_data = (
    job_skills.groupby(["category", "skill_name"])
      .size()
      .reset_index(name="count")
)
heat_data["rate"] = heat_data.apply(
    lambda r: r["count"] / cat_totals.get(r["category"], 1) * 100, axis=1
)
heat_pivot = heat_data.pivot(index="skill_name", columns="category", values="rate").fillna(0)

fig = go.Figure(go.Heatmap(
    z=heat_pivot.values,
    x=heat_pivot.columns.tolist(),
    y=heat_pivot.index.tolist(),
    colorscale="Blues",
    text=np.round(heat_pivot.values, 1),
    texttemplate="%{text}%",
    colorbar=dict(title="% of Postings")
))
fig.update_layout(
    title="LinkedIn Skill Demand Rate by CS Subcategory",
    xaxis_title="CS Subcategory",
    yaxis_title="Skill Category",
    height=700
)
fig.show()

### 3b. Specific Tech Skills (keyword extraction from skills_desc)

In [ ]:
TECH_SKILLS = [
    # Languages
    "python", "java", "javascript", "typescript", "c++", "c#", "golang",
    "rust", "scala", "kotlin", "swift", "ruby", "php", "bash",
    # Web
    "react", "angular", "vue", "node.js", "django", "flask", "fastapi", "spring",
    # Data & ML
    "sql", "spark", "kafka", "airflow", "dbt", "pandas", "numpy",
    "scikit-learn", "tensorflow", "pytorch", "tableau", "power bi", "looker",
    # Cloud & Infra
    "aws", "azure", "gcp", "google cloud", "kubernetes", "docker", "terraform",
    "ansible", "jenkins", "linux", "git",
    # Databases
    "postgresql", "mysql", "mongodb", "redis", "elasticsearch",
    "snowflake", "redshift", "bigquery", "dynamodb",
    # Concepts
    "rest api", "graphql", "microservices", "ci/cd", "agile",
]

def extract_tech_skills(text: str) -> list:
    return [s for s in TECH_SKILLS if s in text]

cs = cs.copy()
cs["tech_skills"] = cs["skills_desc"].apply(extract_tech_skills)

tech_rows = (
    cs[["job_id", "category", "month", "tech_skills"]]
      .explode("tech_skills")
      .dropna(subset=["tech_skills"])
      .rename(columns={"tech_skills": "skill"})
)

print(f"Tech skill mentions: {len(tech_rows):,}")
print(f"Jobs with ≥1 tech skill: {tech_rows['job_id'].nunique():,}")

In [ ]:
# Top 25 tech skills — horizontal bar
top25 = tech_rows["skill"].value_counts().head(25).reset_index()

fig = px.bar(
    top25[::-1],
    x="count", y="skill",
    orientation="h",
    title="Top 25 Tech Skills in CS Job Postings (skills_desc)",
    labels={"skill": "Skill", "count": "Postings Mentioning Skill"},
    color="count",
    color_continuous_scale="Blues"
)
fig.update_layout(coloraxis_showscale=False, height=700)
fig.show()

In [ ]:
# Tech skill mention rate over time (top 10)
top10 = tech_rows["skill"].value_counts().head(10).index.tolist()
monthly_totals = cs.groupby("month").size().rename("total")

skill_trend = (
    tech_rows[tech_rows["skill"].isin(top10)]
      .groupby(["month", "skill"])
      .size()
      .reset_index(name="count")
      .merge(monthly_totals, on="month")
)
skill_trend["rate"] = skill_trend["count"] / skill_trend["total"] * 100

fig = px.line(
    skill_trend.sort_values("month"),
    x="month", y="rate",
    color="skill",
    markers=True,
    title="Top 10 Tech Skills — Monthly Mention Rate (% of CS Postings)",
    labels={"month": "Month", "rate": "Mention Rate (%)", "skill": "Skill"},
    color_discrete_sequence=px.colors.qualitative.Vivid
)
fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
# Tech skill co-occurrence matrix (top 15)
top15 = tech_rows["skill"].value_counts().head(15).index.tolist()
pivot = (
    tech_rows[tech_rows["skill"].isin(top15)]
      .assign(v=1)
      .pivot_table(index="job_id", columns="skill", values="v", fill_value=0)
)
cooc = pivot.T.dot(pivot)
cooc_arr = cooc.to_numpy().copy()
np.fill_diagonal(cooc_arr, 0)

fig = go.Figure(go.Heatmap(
    z=cooc_arr,
    x=cooc.columns.tolist(),
    y=cooc.index.tolist(),
    colorscale="Purples",
    colorbar=dict(title="Co-occurrences")
))
fig.update_layout(
    title="Tech Skill Co-occurrence Matrix (Top 15)",
    xaxis_tickangle=45
)
fig.show()

## 4. Salary & Compensation
Uses the pre-normalized annual salary column (`normalized_salary`).

In [ ]:
sal = cs.dropna(subset=["normalized_salary"]).copy()
sal = sal[(sal["normalized_salary"] >= 20_000) & (sal["normalized_salary"] <= 600_000)]
print(f"CS postings with salary: {len(sal):,}")

In [ ]:
fig = px.box(
    sal,
    x="category", y="normalized_salary",
    color="category",
    title="Annual Salary Distribution by CS Subcategory",
    labels={"category": "Subcategory", "normalized_salary": "Annual Salary ($)"},
    color_discrete_sequence=px.colors.qualitative.Bold,
    points="outliers"
)
fig.update_layout(showlegend=False)
fig.update_yaxes(tickprefix="$", tickformat=",.0f")
fig.show()

In [ ]:
# Salary by remote vs on-site
sal["work_mode"] = sal["remote_allowed"].map({1.0: "Remote", 0.0: "On-site"}).fillna("Unspecified")

fig = px.violin(
    sal[sal["work_mode"] != "Unspecified"],
    x="work_mode", y="normalized_salary",
    color="work_mode",
    box=True,
    title="Salary Distribution: Remote vs On-site",
    labels={"work_mode": "Work Mode", "normalized_salary": "Annual Salary ($)"},
    color_discrete_map={"Remote": "royalblue", "On-site": "tomato"}
)
fig.update_layout(showlegend=False)
fig.update_yaxes(tickprefix="$", tickformat=",.0f")
fig.show()

In [ ]:
# Median salary trend over time
sal_monthly = (
    sal.groupby("month")["normalized_salary"]
      .median()
      .reset_index()
      .sort_values("month")
)

fig = px.line(
    sal_monthly,
    x="month", y="normalized_salary",
    markers=True,
    title="Median CS Job Salary Over Time",
    labels={"month": "Month", "normalized_salary": "Median Annual Salary ($)"}
)
fig.update_yaxes(tickprefix="$", tickformat=",.0f")
fig.update_xaxes(tickangle=45)
fig.show()

## 5. Work Type & Remote Trends

In [ ]:
wt_counts = cs["formatted_work_type"].value_counts().reset_index()

fig = px.pie(
    wt_counts,
    names="formatted_work_type", values="count",
    title="CS Jobs by Work Type (Full-time / Part-time / Contract …)",
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig.show()

In [ ]:
cs_r = cs.copy()
cs_r["work_mode"] = cs_r["remote_allowed"].map({1.0: "Remote", 0.0: "On-site"}).fillna("Unspecified")

remote_monthly = (
    cs_r.groupby(["month", "work_mode"])
      .size()
      .reset_index(name="count")
      .sort_values("month")
)

fig = px.line(
    remote_monthly,
    x="month", y="count",
    color="work_mode",
    markers=True,
    title="Remote vs On-site CS Job Listings Over Time",
    labels={"month": "Month", "count": "Postings", "work_mode": "Work Mode"},
    color_discrete_map={"Remote": "royalblue", "On-site": "tomato", "Unspecified": "gray"}
)
fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
remote_rate = (
    cs.groupby("category")["remote_allowed"]
      .mean()
      .mul(100)
      .reset_index()
      .rename(columns={"remote_allowed": "remote_pct"})
      .sort_values("remote_pct")
)

fig = px.bar(
    remote_rate,
    x="remote_pct", y="category",
    orientation="h",
    title="Remote Work Rate by CS Subcategory",
    labels={"remote_pct": "% Postings Allowing Remote", "category": "Subcategory"},
    color="remote_pct",
    color_continuous_scale="Blues"
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

## 6. Experience Level & Geography

In [ ]:
exp = (
    cs[cs["formatted_experience_level"].notna()]
      .groupby(["category", "formatted_experience_level"])
      .size()
      .reset_index(name="count")
)

fig = px.bar(
    exp,
    x="category", y="count",
    color="formatted_experience_level",
    barmode="stack",
    title="Experience Level Distribution by CS Subcategory",
    labels={"category": "Subcategory", "count": "Postings", "formatted_experience_level": "Level"},
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.show()

In [ ]:
# US choropleth by state (from company state column)
state_counts = (
    cs[cs["state"].notna() & cs["state"].str.len().eq(2)]
      .groupby("state")
      .size()
      .reset_index(name="count")
)

fig = px.choropleth(
    state_counts,
    locations="state",
    locationmode="USA-states",
    color="count",
    scope="usa",
    title="CS Job Postings by US State",
    color_continuous_scale="Blues",
    labels={"count": "Postings"}
)
fig.show()

In [ ]:
# Top 15 cities
top_cities = (
    cs[cs["city"].notna()]
      .groupby("city")
      .size()
      .reset_index(name="count")
      .sort_values("count", ascending=False)
      .head(15)
)

fig = px.bar(
    top_cities[::-1],
    x="count", y="city",
    orientation="h",
    title="Top 15 Cities for CS Job Postings",
    labels={"city": "City", "count": "Postings"},
    color="count",
    color_continuous_scale="Teal"
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

## 7. Company & Demand Stats

In [ ]:
top_companies = (
    cs[cs["company_name"].notna()]
      .groupby("company_name")
      .size()
      .reset_index(name="count")
      .sort_values("count", ascending=False)
      .head(20)
)

fig = px.bar(
    top_companies[::-1],
    x="count", y="company_name",
    orientation="h",
    title="Top 20 Companies by CS Job Postings",
    labels={"company_name": "Company", "count": "Postings"},
    color="count",
    color_continuous_scale="Oranges"
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [ ]:
# Apply-to-view rate by subcategory
engagement = (
    cs.dropna(subset=["views", "applies"])
      .assign(ratio=lambda df: df["applies"] / df["views"].clip(lower=1))
      .groupby("category")["ratio"]
      .median()
      .reset_index()
      .rename(columns={"ratio": "apply_rate"})
      .sort_values("apply_rate")
)

fig = px.bar(
    engagement,
    x="apply_rate", y="category",
    orientation="h",
    title="Median Apply-to-View Rate by CS Subcategory",
    labels={"apply_rate": "Applies / Views (median)", "category": "Subcategory"},
    color="apply_rate",
    color_continuous_scale="Greens"
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [ ]:
# Company size distribution for CS postings
size_labels = {
    1: "1", 2: "2–10", 3: "11–50", 4: "51–200",
    5: "201–500", 6: "501–1K", 7: "1K–5K", 8: "5K–10K", 9: "10K+"
}

size_dist = (
    cs[cs["company_size"].notna()]
      .assign(size_label=lambda df: df["company_size"].map(size_labels))
      .groupby(["company_size", "size_label"])
      .size()
      .reset_index(name="count")
      .sort_values("company_size")
)

fig = px.bar(
    size_dist,
    x="size_label", y="count",
    title="CS Job Postings by Company Size",
    labels={"size_label": "Company Size (employees)", "count": "Postings"},
    color="count",
    color_continuous_scale="Oranges"
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [ ]:
# Posting duration histogram
cs_dur = cs.dropna(subset=["expiry_dt"]).copy()
cs_dur["duration_days"] = (cs_dur["expiry_dt"] - cs_dur["listed_dt"]).dt.days
cs_dur = cs_dur[(cs_dur["duration_days"] > 0) & (cs_dur["duration_days"] < 365)]

fig = px.histogram(
    cs_dur,
    x="duration_days",
    color="category",
    nbins=60,
    barmode="overlay",
    opacity=0.65,
    title="Job Posting Duration (Days Until Expiry) by CS Subcategory",
    labels={"duration_days": "Duration (days)", "category": "Category"},
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig.show()

---
*End of analysis. All charts are interactive — hover, zoom, and click legend items to filter.*